# Preprocessing of data

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [9]:
ds = pd.read_csv("bank-additional-full.csv")

## missing/false data

In [10]:
# Normalize column names to lowercase
ds_norm = ds.rename(columns=lambda col: col.strip().lower())

# ---------------------------------------------------------
# 1. Detection: Missing Values (NaN / None / pd.NA)
# ---------------------------------------------------------
mask_missing = ds_norm.isna().any(axis=1)
ds_missing = ds[mask_missing].copy()

# ---------------------------------------------------------
# 2. Detection: Duplicate Rows
# ---------------------------------------------------------
# Redundant duplicates (excluding the initial occurrence)
mask_duplicates_redundant = ds_norm.duplicated(keep="first")
num_duplicates_redundant = int(mask_duplicates_redundant.sum())

# Total rows participating in duplicate groups (including the first occurrence)
mask_duplicates_all = ds_norm.duplicated(keep=False)
num_duplicates_all = int(mask_duplicates_all.sum())

# Extracted duplicate subsets
ds_duplicates = ds[mask_duplicates_redundant & (~mask_missing)].copy()
ds_missing_or_duplicate = ds[(mask_missing | mask_duplicates_redundant)].copy()

# ---------------------------------------------------------
# 3. Domain Specifications
# ---------------------------------------------------------
VALID_CATEGORIES = {
    "job": {
        "admin.", "blue-collar", "entrepreneur", "housemaid", "management",
        "retired", "self-employed", "services", "student", "technician",
        "unemployed", "unknown"
    },
    "marital": {"divorced", "married", "single", "unknown"},
    "education": {
        "basic.4y", "basic.6y", "basic.9y", "high.school", "illiterate",
        "professional.course", "university.degree", "unknown"
    },
    "default": {"no", "yes", "unknown"},
    "housing": {"no", "yes", "unknown"},
    "loan": {"no", "yes", "unknown"},
    "contact": {"cellular", "telephone"},
    "month": {
        "jan", "feb", "mar", "apr", "may", "jun",
        "jul", "aug", "sep", "oct", "nov", "dec"
    },
    "day_of_week": {"mon", "tue", "wed", "thu", "fri"},
    "poutcome": {"failure", "nonexistent", "success"},
    "y": {"yes", "no"}
}

NUMERIC_BOUNDS = {
    "age": lambda s: (s <= 0) | (s > 120),
    "duration": lambda s: s < 0,
    "campaign": lambda s: s < 1,
    "pdays": lambda s: (s < 0) | (s > 999),
    "previous": lambda s: s < 0,
    "nr.employed": lambda s: s <= 0,
    "cons.price.idx": lambda s: s <= 0
}

# ---------------------------------------------------------
# 4. Detection: False / Invalid Domain Records
# ---------------------------------------------------------
mask_cat_error = pd.Series(False, index=ds.index)
for col, allowed_values in VALID_CATEGORIES.items():
    if col in ds_norm.columns:
        series = ds_norm[col]
        is_invalid = series.notna() & (
            ~series.astype(str).str.strip().str.lower().isin(allowed_values)
        )
        mask_cat_error |= is_invalid

mask_num_error = pd.Series(False, index=ds.index)
for col, constraint in NUMERIC_BOUNDS.items():
    if col in ds_norm.columns:
        series = ds_norm[col]
        coerced = pd.to_numeric(series, errors="coerce")
        type_error = series.notna() & coerced.isna()
        range_error = coerced.notna() & constraint(coerced)
        mask_num_error |= (type_error | range_error)

# Exclude rows already isolated by missing or duplicate filters
mask_invalid = (mask_cat_error | mask_num_error) & (~(mask_missing | mask_duplicates_redundant))
ds_invalid = ds[mask_invalid].copy()

# ---------------------------------------------------------
# 5. Clean Partition
# ---------------------------------------------------------
ds_clean = ds[~(mask_missing | mask_duplicates_redundant | mask_invalid)].copy()

# ---------------------------------------------------------
# 6. Summary Output
# ---------------------------------------------------------
print(f"Total records:                {len(ds)}")
print(f"Total duplicates (redundant): {num_duplicates_redundant}")
print(f"Total rows in duplicate sets: {num_duplicates_all}")
print(f"Missing (NaN) rows:           {len(ds_missing)}")
print(f"Invalid / out-of-bounds rows: {len(ds_invalid)}")
print(f"Clean, unique records:        {len(ds_clean)}")

Total records:                41188
Total duplicates (redundant): 12
Total rows in duplicate sets: 24
Missing (NaN) rows:           0
Invalid / out-of-bounds rows: 0
Clean, unique records:        41176


We didn't find any wrong values (spelling errors, negative numbers etc)
As every numerical value can be generated automatically without human interference, we dont expect unknown values there (or categorically placed in one slot)
12 duplicate values were found and removed

## 80/20 Split of data in train and test

In [11]:
ds_train, ds_test = train_test_split(
    ds_clean,
    test_size=0.20,
    random_state=69
)

## Export test and train

In [15]:
ds_train.to_csv("ds_train.csv", index=False)
ds_test.to_csv("ds_test.csv", index=False)